In [ ]:
"""
Maak jaarlijkse Ember prijsdatasets uit de volledige Belgium.csv
=================================================================
Filtert een opgegeven jaar uit de Ember dataset, gebruikt Datetime (Local)
als tijdreferentie. Prijzen worden bewaard in EUR/MWh (marktconventie),
consistent met de tariefformules in simulatie.py en simulatie_batterij.py.
Dit script werd gebruikt om zowel Belgium_2024_MWh.csv (referentiejaar)
als Belgium_2022_MWh.csv (crisisscenario) aan te maken.
Gebruik:
    python maak_ember_dataset.py
Input:  Belgium.csv  (Ember European Wholesale Electricity Price Data)
Output: Belgium_{jaar}_MWh.csv  voor elk opgegeven jaar
"""
import pandas as pd

# ── Parameters ───────────────────────────────────────────────
BESTANDSPAD = "Belgium.csv"
JAREN       = [2022, 2024]   # Voeg jaren toe of verwijder naar wens

# ── Laad data ────────────────────────────────────────────────
df = pd.read_csv(BESTANDSPAD)
df.columns = ['Country', 'ISO3 Code', 'Datetime (UTC)', 'Datetime (Local)', 'Price (EUR/MWh)']
df['Datetime (Local)'] = pd.to_datetime(df['Datetime (Local)'])

# ── Maak dataset per jaar ────────────────────────────────────
for jaar in JAREN:
    subset = df[df['Datetime (Local)'].dt.year == jaar].copy()
    subset = subset[['Datetime (Local)', 'Price (EUR/MWh)']].reset_index(drop=True)
    uitvoerbestand = f"Belgium_{jaar}_MWh.csv"
    subset.to_csv(uitvoerbestand, index=False)
    print(f"\n{jaar}: {len(subset)} uren → {uitvoerbestand}")
    print(f"  Van : {subset['Datetime (Local)'].min()}")
    print(f"  Tot : {subset['Datetime (Local)'].max()}")
    print(f"  Gem : {subset['Price (EUR/MWh)'].mean():.1f} EUR/MWh")
    print(f"  Std : {subset['Price (EUR/MWh)'].std():.1f} EUR/MWh")
    print(f"  Min : {subset['Price (EUR/MWh)'].min():.1f} EUR/MWh")
    print(f"  Max : {subset['Price (EUR/MWh)'].max():.1f} EUR/MWh")
    n_neg = (subset['Price (EUR/MWh)'] < 0).sum()
    print(f"  < 0 : {n_neg} uren ({n_neg/len(subset)*100:.1f}%)")
    print(f"  ✓ Opgeslagen: {uitvoerbestand}")

In [ ]:
"""
============================================================
VOORBEREIDING — KWARTIERDATA NAAR UURDATA
============================================================
De ruwe Fluvius open data bevat metingen per kwartier.
Dit script aggregeert die naar uurwaarden door de volumes
per uur op te tellen. Indicatoren (PV, WP, EV) en het
contracttype worden overgenomen van het eerste kwartier.

Eenmalig uitvoeren per profielbestand vóór simulatie.py.

Pas INPUT_BESTAND en OUTPUT_BESTAND aan per profieltype.

Voorbeelden:
  Input  → P6269_Open_Data_enkel_ZP.csv
         → P6269_Open_Data_geen_ZP.csv
         → P6269_Open_Data_WP_geen_ZP.csv
         → P6269_Open_Data_WP_met_ZP.csv
         → P6269_Open_Data_EV_geen_ZP.csv
         → P6269_Open_Data_EV_met_ZP.csv
         → P6269_Open_Data_WP_EV_geen_ZP.csv
         → P6269_Open_Data_WP_EV_met_ZP.csv
  Output → fluvius_300_met_ZP_uur.csv
         → fluvius_300_zonder_middelen_uur.csv
         → fluvius_300_met_WP_uur.csv
         → fluvius_300_met_WP_met_ZP_uur.csv
         → fluvius_300_met_EV_uur.csv
         → fluvius_300_met_EV_met_ZP_uur.csv
         → fluvius_300_met_WP_met_EV_uur.csv
         → fluvius_300_met_WP_met_EV_met_ZP_uur.csv

Input:
  <profielbestand>.csv  (Fluvius kwartierdata, 300 huishoudens)

Output:
  fluvius_300_<profiel>_uur.csv
============================================================
"""

import pandas as pd

# ── Parameters — pas aan per profielbestand ──────────────────

INPUT_BESTAND  = "P6269_Open_Data_WP_EV_met_ZP.csv"
OUTPUT_BESTAND = "fluvius_300_met_WP_met_EV_met_ZP_uur.csv"

# ── Inlezen ──────────────────────────────────────────────────

print(f"Inlezen: {INPUT_BESTAND}")
df = pd.read_csv(INPUT_BESTAND)
print(f"  {len(df):,} rijen | {df['EAN_ID'].nunique()} huishoudens")

# Tijdstempel inlezen met UTC-flag — de Fluvius kwartierdata staat
# in UTC (herkenbaar aan de Z-suffix, bv. 2024-03-27T02:00:00.000Z).
# De UTC-info wordt bewaard zodat de aggregatie op de juiste uren
# gebeurt en simulatie.py de tijdstempel correct kan omzetten naar
# Belgische lokale tijd.
df["Datum_Startuur"] = pd.to_datetime(df["Datum_Startuur"], utc=True)

# Afronding naar het uur: vier kwartierwaarden per uur worden samengevoegd.
# De afronding gebeurt in UTC zodat de zomertijdovergang geen problemen geeft.
df["Uur"] = df["Datum_Startuur"].dt.floor("h")

# ── Aggregatie kwartier → uur ─────────────────────────────────

# Volumes worden opgeteld (kWh per kwartier → kWh per uur).
# Indicatoren en contracttype zijn constant per huishouden en
# worden overgenomen van het eerste kwartier van elk uur.
df_uur = df.groupby(["EAN_ID", "Datum", "Uur"]).agg(
    Volume_Afname_KWh              = ("Volume_Afname_KWh",              "sum"),
    Volume_Injectie_KWh            = ("Volume_Injectie_KWh",            "sum"),
    Warmtepomp_Indicator           = ("Warmtepomp_Indicator",           "first"),
    Elektrisch_Voertuig_Indicator  = ("Elektrisch_Voertuig_Indicator",  "first"),
    PV_Installatie_Indicator       = ("PV_Installatie_Indicator",       "first"),
    Contract_Categorie             = ("Contract_Categorie",             "first"),
).reset_index()

# Kolom hernoemen zodat de structuur identiek blijft aan de inputbestanden
df_uur = df_uur.rename(columns={"Uur": "Datum_Startuur"})
df_uur = df_uur[[
    "EAN_ID", "Datum", "Datum_Startuur",
    "Volume_Afname_KWh", "Volume_Injectie_KWh",
    "Warmtepomp_Indicator", "Elektrisch_Voertuig_Indicator",
    "PV_Installatie_Indicator", "Contract_Categorie",
]]

# ── Verificatie ───────────────────────────────────────────────

verwacht_uren = df["EAN_ID"].nunique() * 365 * 24
uren_per_hh   = df_uur.groupby("EAN_ID").size()
print(f"\n  Uren na aggregatie: {len(df_uur):,} (verwacht: {verwacht_uren:,})")
print(f"  Uren per huishouden: min {uren_per_hh.min()}, max {uren_per_hh.max()}, "
      f"gem {uren_per_hh.mean():.0f}")

# ── Opslaan ───────────────────────────────────────────────────

df_uur.to_csv(OUTPUT_BESTAND, index=False)
print(f"\n✓ Opgeslagen: {OUTPUT_BESTAND}")

In [ ]:
"""
============================================================
VOORBEREIDING — STEEKPROEVEN 1.000 HUISHOUDENS (ALLE SCENARIO'S)
============================================================
Trekt voor elk scenario een representatieve steekproef van
1.000 huishoudens uit de Fluvius-profielbestanden, gewogen
naar de Vlaamse penetratiegraden van zonnepanelen (PV),
warmtepompen (WP) en elektrische voertuigen (EV).

De aantallen per profieltype worden bepaald via de largest
remainder method zodat ze optellen tot exact 1.000.
Wanneer een profieltype meer huishoudens nodig heeft dan
er unieke meters beschikbaar zijn (300), wordt met
teruglegging getrokken.

Eenmalig uitvoeren vóór simulatie.py.

Input:
  fluvius_300_<profiel>_uur.csv  (uurdata, 8 profielbestanden,
  output van prep_fluvius_kwartier_naar_uur.py)

Output (één CSV per scenario):
  dataset_1000_huishoudens.csv                          ← 2024
  dataset_1000_huishoudens_2034.csv                     ← 2034
  dataset_1000_huishoudens_2034_highEV.csv              ← High EV
  dataset_1000_huishoudens_volledige_elektrificatie.csv
============================================================
"""

# Importeer de benodigde bibliotheken
import pandas as pd   # voor het lezen, bewerken en opslaan van CSV-bestanden
import numpy as np    # voor wiskundige bewerkingen en willekeurige steekproeftrekking
import os             # voor het samenvoegen van mapnamen en bestandsnamen

# ── Algemene parameters ───────────────────────────────────────

DATA_DIR    = "."    # map waar de inputbestanden staan (huidige map)
RANDOM_SEED = 42     # vaste seed zodat de steekproef bij elke run identiek is (reproduceerbaarheid)
TOTAAL      = 1_000  # totaal aantal huishoudens in elke steekproef

# Koppeling tussen profielnaam en bijhorend Fluvius-bestand
# Elke sleutel is een interne profielnaam; elke waarde is de bestandsnaam op schijf
BESTANDEN = {
    "zonder_middelen":      "fluvius_300_zonder_middelen_uur.csv",
    "met_ZP":               "fluvius_300_met_ZP_uur.csv",
    "met_WP":               "fluvius_300_met_WP_uur.csv",
    "met_EV":               "fluvius_300_met_EV_uur.csv",
    "met_WP_met_ZP":        "fluvius_300_met_WP_met_ZP_uur.csv",
    "met_EV_met_ZP":        "fluvius_300_met_EV_met_ZP_uur.csv",
    "met_WP_met_EV":        "fluvius_300_met_WP_met_EV_uur.csv",
    "met_WP_met_EV_met_ZP": "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
}

# ── Scenario's: penetratiegraden + outputbestand ─────────────
#
# ZP = aandeel huishoudens met zonnepanelen
# WP = aandeel huishoudens met warmtepomp
# EV = aandeel huishoudens met elektrisch voertuig
#
# De penetratiegraden worden verondersteld onafhankelijk te zijn,
# zodat het gewicht van elk profieltype het product is van de
# afzonderlijke kansen (bv. PV + EV = ZP × (1-WP) × EV).

# Elk scenario bevat de drie penetratiegraden en de naam van het outputbestand
SCENARIOS = {
    "Baseline 2024": {
        "ZP": 0.31, "WP": 0.03, "EV": 0.10,
        "output": "dataset_1000_huishoudens.csv",
    },
    "2034": {
        "ZP": 0.50, "WP": 0.15, "EV": 0.45,
        "output": "dataset_1000_huishoudens_2034.csv",
    },
    "High EV": {
        "ZP": 0.50, "WP": 0.15, "EV": 0.80,
        "output": "dataset_1000_huishoudens_2034_highEV.csv",
    },
    "Volledige elektrificatie": {
        "ZP": 0.80, "WP": 0.60, "EV": 0.85,
        "output": "dataset_1000_huishoudens_volledige_elektrificatie.csv",
    },
}


# ── Hulpfunctie: aantallen bepalen via largest remainder method ──

def bereken_aantallen(ZP: float, WP: float, EV: float, totaal: int) -> dict:
    """
    Berekent het aantal huishoudens per profieltype zodat de verdeling
    de opgegeven penetratiegraden zo goed mogelijk benadert en de
    aantallen optellen tot exact `totaal`.

    De largest remainder method wijst resterende plaatsen toe aan de
    profieltypes met de grootste fractionele rest na afronding naar beneden.
    """
    # Bereken het theoretische gewicht van elk profieltype via de onafhankelijkheidsaanname:
    # de kans op een combinatie = product van de afzonderlijke kansen
    # Bv. "met_ZP" = kans dat iemand WEL ZP heeft, GEEN WP en GEEN EV
    gewichten = {
        "zonder_middelen":      (1-ZP) * (1-WP) * (1-EV),   # geen van de drie technologieën
        "met_ZP":               ZP     * (1-WP) * (1-EV),   # alleen zonnepanelen
        "met_WP":               (1-ZP) * WP     * (1-EV),   # alleen warmtepomp
        "met_EV":               (1-ZP) * (1-WP) * EV,       # alleen elektrisch voertuig
        "met_WP_met_ZP":        ZP     * WP     * (1-EV),   # zonnepanelen + warmtepomp
        "met_EV_met_ZP":        ZP     * (1-WP) * EV,       # zonnepanelen + elektrisch voertuig
        "met_WP_met_EV":        (1-ZP) * WP     * EV,       # warmtepomp + elektrisch voertuig
        "met_WP_met_EV_met_ZP": ZP     * WP     * EV,       # alle drie de technologieën
    }

    # Normaliseer: som van alle gewichten = 1 (voor de zekerheid, door deling)
    totaal_gewicht = sum(gewichten.values())

    # Bereken het exacte (niet-gehele) aantal huishoudens per profieltype
    # Bv. als gewicht = 0.27 en totaal = 1000, dan exact = 270.0
    exact = {p: (g / totaal_gewicht) * totaal for p, g in gewichten.items()}

    # Rond alle exacte aantallen naar beneden af naar gehele getallen
    # Bv. 270.7 wordt 270
    aantallen = {p: int(v) for p, v in exact.items()}

    # Bereken hoeveel plaatsen er nog ontbreken door de afrondingen
    # Bv. als de som 997 is en totaal 1000, dan verschil = 3
    verschil = totaal - sum(aantallen.values())

    # Verdeel de resterende plaatsen aan de profieltypes met de grootste decimale rest
    # Bv. 270.7 heeft rest 0.7, 130.2 heeft rest 0.2 → 270.7 krijgt een extra huishouden eerst
    for profiel in sorted(exact, key=lambda p: exact[p] % 1, reverse=True)[:verschil]:
        aantallen[profiel] += 1  # voeg één extra huishouden toe aan dit profieltype

    return aantallen  # geeft een dict terug met het definitieve aantal per profieltype


# ── Hulpfunctie: steekproef trekken voor één scenario ────────

def maak_steekproef(scenario_naam: str, ZP: float, WP: float, EV: float,
                    output_bestand: str) -> None:
    """
    Trekt een steekproef van 1.000 huishoudens voor het opgegeven scenario
    en slaat het resultaat op als CSV.

    Voor profieltypes waarvoor minder dan het benodigde aantal unieke meters
    beschikbaar is, wordt met teruglegging getrokken. Elk getrokken huishouden
    krijgt een uniek EAN_ID zodat duplicaten in de dataset geen problemen
    veroorzaken bij de latere groepering in simulatie.py.
    """
    # Print een visuele scheiding en de naam van het huidige scenario
    print(f"\n{'─' * 55}")
    print(f"Scenario: {scenario_naam}  (ZP={ZP*100:.0f}%, WP={WP*100:.0f}%, EV={EV*100:.0f}%)")
    print(f"{'─' * 55}")

    # Bereken hoeveel huishoudens per profieltype nodig zijn voor dit scenario
    aantallen = bereken_aantallen(ZP, WP, EV, TOTAAL)

    # Print een overzicht van de aantallen per profieltype
    for profiel, aantal in aantallen.items():
        print(f"  {profiel:<30} {aantal:>4} ({aantal/TOTAAL*100:.1f}%)")
    print(f"  {'TOTAAL':<30} {sum(aantallen.values()):>4}")

    # Laad alle benodigde profielbestanden eenmalig in een lookup-dictionary
    # zodat we niet per getrokken EAN opnieuw van schijf hoeven te lezen
    # Structuur: ean_lookup[profiel][ean_id] = DataFrame met alle rijen van die meter
    ean_lookup = {}
    for profiel, aantal in aantallen.items():
        if aantal == 0:
            continue  # sla profieltypes over die niet nodig zijn in dit scenario

        # Bouw het volledige pad naar het profielbestand
        pad = os.path.join(DATA_DIR, BESTANDEN[profiel])

        # Lees het CSV-bestand in als een pandas DataFrame
        df = pd.read_csv(pad)

        # Verwijder een eventuele eerste kolom met rij-indices (artefact van eerdere opslag)
        if df.columns[0] not in ["EAN_ID", "Datum"]:
            df = df.iloc[:, 1:]

        # Groepeer alle rijen per meter (EAN_ID) en sla op in de lookup-dictionary
        # Zo kunnen we later snel alle rijen van één specifieke meter ophalen
        ean_lookup[profiel] = {ean: groep.copy() for ean, groep in df.groupby("EAN_ID")}

    # Maak een willekeurige getalsgenerator met vaste seed voor reproduceerbaarheid
    rng = np.random.default_rng(RANDOM_SEED)

    # Lijst om alle getrokken huishoudens (als DataFrames) in te verzamelen
    alle_delen = []

    for profiel, aantal in aantallen.items():
        if aantal == 0:
            continue  # sla profieltypes over die niet nodig zijn

        # Haal alle beschikbare EAN-nummers op voor dit profieltype
        beschikbaar   = list(ean_lookup[profiel].keys())
        n_beschikbaar = len(beschikbaar)  # typisch 300 unieke meters per profieltype

        # Bepaal of teruglegging nodig is:
        # als we meer huishoudens nodig hebben dan er unieke meters zijn,
        # moeten we dezelfde meter meerdere keren gebruiken
        replace = aantal > n_beschikbaar
        if replace:
            # Waarschuw de gebruiker dat teruglegging wordt gebruikt
            print(f"  ! {profiel}: {aantal} nodig, {n_beschikbaar} beschikbaar → teruglegging")

        # Trek willekeurig het gewenste aantal EAN-nummers uit de beschikbare pool
        # replace=True: een meter mag meerdere keren worden gekozen
        # replace=False: elk meter wordt maximaal één keer gekozen
        gekozen = rng.choice(beschikbaar, size=aantal, replace=replace)

        # Maak voor elk gekozen EAN-nummer een kopie van de rijen en ken een uniek ID toe
        rijen_lijst = []
        for i, ean in enumerate(gekozen):
            # Haal alle uurdata op voor dit specifieke EAN-nummer
            rijen = ean_lookup[profiel][ean].copy()

            # Ken een uniek EAN_ID toe dat ook bij teruglegging uniek blijft:
            # het bevat de profielnaam, een volgnummer (i) en het originele EAN
            rijen["EAN_ID"]        = f"{profiel}_sample_{i}_orig_{int(ean)}"

            # Bewaar het originele EAN-nummer voor traceerbaarheid
            rijen["Origineel_EAN"] = int(ean)

            # Voeg de profielnaam toe als extra kolom
            rijen["Profiel"]       = profiel

            rijen_lijst.append(rijen)

        # Voeg alle rijen van dit profieltype samen tot één DataFrame
        alle_delen.append(pd.concat(rijen_lijst, ignore_index=True))

    # Voeg alle profieltypes samen tot één volledige dataset van 1.000 huishoudens
    dataset = pd.concat(alle_delen, ignore_index=True)

    # Hernoem de EAN_IDs naar eenvoudige gehele getallen (1 t/m 1000)
    # zodat elke huishoud een uniek en leesbaar nummer krijgt
    unieke_eans = dataset["EAN_ID"].unique()           # lijst van alle unieke EAN-strings
    mapping     = {ean: i + 1 for i, ean in enumerate(unieke_eans)}  # koppel elke string aan een getal
    dataset["EAN_ID"] = dataset["EAN_ID"].map(mapping) # vervang de strings door de getallen

    # Sla de volledige dataset op als CSV-bestand
    uitvoer = os.path.join(DATA_DIR, output_bestand)
    dataset.to_csv(uitvoer, index=False)  # index=False: geen extra rij-nummers in het bestand

    # Print een bevestiging met het aantal rijen en unieke huishoudens
    print(f"\n  ✓ Opgeslagen: {output_bestand}")
    print(f"    Rijen: {len(dataset):,}  |  Huishoudens: {dataset['EAN_ID'].nunique()}")


# ── Hoofdprogramma: alle scenario's uitvoeren ─────────────────

print("STEEKPROEFTREKKING — ALLE SCENARIO'S")

# Loop over elk scenario en roep de steekproeffunctie aan
for naam, config in SCENARIOS.items():
    maak_steekproef(
        scenario_naam  = naam,           # naam van het scenario (voor de output)
        ZP             = config["ZP"],   # penetratiegraad zonnepanelen
        WP             = config["WP"],   # penetratiegraad warmtepompen
        EV             = config["EV"],   # penetratiegraad elektrische voertuigen
        output_bestand = config["output"],  # naam van het outputbestand
    )

# Print een afsluitende bevestiging
print(f"\n{'=' * 55}")
print("ALLE STEEKPROEVEN AANGEMAAKT")
print(f"{'=' * 55}")

In [ ]:
"""
============================================================
SIMULATIE — DYNAMISCHE VS VASTE ENERGIECONTRACTEN
Thesis: Death spiral & tariefvergelijking
============================================================
Dit bestand voert de volledige analyse uit, van ruwe
inputbestanden tot eindresultaten.

VOLGORDE:
  1. EPEX-prijzen laden (Ember, 2024 + 2022)
  2. Individuele kostenpositie — alle 2.400 Fluvius-meters
       - Individueel leverancierstarief per klant (excl. btw)
       - Kostenkwartielen: welke profielen zijn goedkoop/duur
       Basis voor puntenwolk en kwartielverdeling plots.
  3. Fluvius-steekproef laden en koppelen aan EPEX
  4. Tarieven berekenen per klant (1.000-steekproef)
       - Individueel leverancierstarief (excl. btw, rangschikking)
       - Portfoliogemiddelde (= leverancierstariefniveau excl. btw)
       - Dynamisch tarief per uur (Eneco Zon & Wind jan 2024, excl. btw)
       Alles vanuit leveranciersperspectief excl. btw. Voor de volledige
       klantfactuur incl. btw op basis van echte tariefkaarten: simulatie_batterij.py
  5. Death spiral simuleren
       - Rangschikking op leveranciersinkoopkost (excl. btw)
       - Tariefevolutie als leverancierstarief excl. btw
       - 4 scenario's 2024 + crisisscenario 2022
  6. Dynamisch vs vast vergelijken — excl. btw
       - Vergelijking leveranciersperspectief
       - 4 scenario's 2024
  7. Gevoeligheidsanalyse prijsvolatiliteit — excl. btw
       - Klantenmix 2034, EPEX 2024 met varierende spreiding

INPUTBESTANDEN (in dezelfde map als dit script):
  Verbruiksdata 1.000-steekproef (output van prep_steekproef.py):
    dataset_1000_huishoudens.csv
    dataset_1000_huishoudens_2034.csv
    dataset_1000_huishoudens_2034_highEV.csv
    dataset_1000_huishoudens_volledige_elektrificatie.csv

  Verbruiksdata 2.400-meters (output van prep_fluvius_kwartier_naar_uur.py):
    fluvius_300_zonder_middelen_uur.csv
    fluvius_300_met_ZP_uur.csv
    fluvius_300_met_WP_uur.csv
    fluvius_300_met_EV_uur.csv
    fluvius_300_met_WP_met_ZP_uur.csv
    fluvius_300_met_EV_met_ZP_uur.csv
    fluvius_300_met_WP_met_EV_uur.csv
    fluvius_300_met_WP_met_EV_met_ZP_uur.csv

  Groothandelsprijzen (output van prep_ember.py):
    Belgium_2024_MWh.csv
    Belgium_2022_MWh.csv

OUTPUTBESTANDEN (CSV):
  individuele_tarieven_2400.csv  - leverancierstarief per klant, alle 2.400 meters
  kostenkwartielen_2400.csv      - profielverdeling per kostenkwartiel
  tarieven_<scenario>.csv        - klantresultaten per scenario (1.000-steekproef)
  spiral_<scenario>.csv          - death spiral curve (energiecomponent, excl. btw)
  spiral_profieldrempels.csv     - wanneer 80% van elk profiel vertrokken is
  dyn_vs_vast.csv                - % dyn beter per profiel en scenario
  gevoeligheid_dyn_vast.csv      - gevoeligheidsanalyse volatiliteit
============================================================
"""

# Importeer de benodigde bibliotheken
import pandas as pd   # voor het lezen, bewerken en opslaan van tabeldata (CSV-bestanden)
import numpy as np    # voor wiskundige bewerkingen zoals percentiel en gemiddelde

# ============================================================
# CONFIGURATIE
# ============================================================

# Risicopremie die de leverancier bovenop de EPEX-inkoopkost rekent (5%)
# Dit dekt het risico dat de leverancier loopt door vooraf een vast tarief vast te leggen
RISICO_PREMIE = 0.05

# BTW wordt niet meegenomen in simulatie.py — alles vanuit leveranciersperspectief excl. btw.
# Zie simulatie_batterij.py voor de volledige klantfactuur incl. btw op basis van Eneco tariefkaarten.

# Tariefparameters dynamisch contract Eneco Zon & Wind (jan 2024, excl. btw)
# Afnametarief per uur (EUR/kWh) = (DYN_ALPHA × EPEX_MWh + DYN_BETA) / 100
DYN_ALPHA     = 0.102   # coëfficiënt op de EPEX-prijs voor afname
DYN_BETA      = 1.0     # vaste opslag bovenop de EPEX-prijs voor afname (ct/kWh)
# Injectievergoeding per uur (EUR/kWh) = (DYN_ALPHA_INJ × EPEX_MWh - DYN_BETA_INJ) / 100
# Let op: kan negatief worden bij lage EPEX-prijzen (klant betaalt dan voor injectie)
DYN_ALPHA_INJ = 0.100   # coëfficiënt op de EPEX-prijs voor injectie
DYN_BETA_INJ  = 1.188   # aftrek op de injectievergoeding (ct/kWh)

# Aantal stappen in de death spiral simulatie
# Elke stap laat 2% extra van de goedkoopste klanten vertrekken (50 stappen = 0% tot 98%)
SPIRAL_STAPPEN  = 50

# Drempelwaarde voor de profielstippellijnen in de death spiral plot
# 0.80 = stippellijn op het punt waarop 80% van een profieltype de portfolio verlaten heeft
DREMPEL_PROFIEL = 0.80

# Gevoeligheidsanalyse: schaalfactoren op de prijsafwijking t.o.v. het jaargemiddelde
# Het jaargemiddelde zelf blijft ongewijzigd; enkel de spreiding (volatiliteit) vergroot
# Factor 1.0 = referentiesituatie (ongewijzigde 2024-prijzen)
# Factor 3.0 is vergelijkbaar met de volatiliteit tijdens de energiecrisis 2022
# (std 134,7 EUR/MWh in 2022 vs 43,0 EUR/MWh in 2024)
VOLATILITEIT_SCENARIOS = {
    "Prijzen 2024 (referentie)":       1.0,   # ongewijzigd
    "Prijzen 2024 +50% volatiliteit":  1.5,   # spreiding 1,5× groter
    "Prijzen 2024 +100% volatiliteit": 2.0,   # spreiding 2× groter
    "Prijzen 2024 +200% volatiliteit": 3.0,   # spreiding 3× groter (vergelijkbaar met 2022)
}

# Koppeling tussen leesbare profielnamen en de bijhorende Fluvius-bestanden (voor 2.400-meters analyse)
FLUVIUS_2400 = {
    "Geen ZP/WP/EV": "fluvius_300_zonder_middelen_uur.csv",
    "ZP":            "fluvius_300_met_ZP_uur.csv",
    "WP":            "fluvius_300_met_WP_uur.csv",
    "EV":            "fluvius_300_met_EV_uur.csv",
    "ZP + WP":       "fluvius_300_met_WP_met_ZP_uur.csv",
    "ZP + EV":       "fluvius_300_met_EV_met_ZP_uur.csv",
    "WP + EV":       "fluvius_300_met_WP_met_EV_uur.csv",
    "ZP + WP + EV":  "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
}

# Koppeling tussen scenario-sleutels en de bijhorende steekproefbestanden (1.000 huishoudens)
FLUVIUS_BESTANDEN = {
    "baseline":       "dataset_1000_huishoudens.csv",
    "2034":           "dataset_1000_huishoudens_2034.csv",
    "highev":         "dataset_1000_huishoudens_2034_highEV.csv",
    "elektrificatie": "dataset_1000_huishoudens_volledige_elektrificatie.csv",
}

# Leesbare labels voor elk scenario (voor gebruik in outputs en prints)
SCENARIO_LABELS = {
    "baseline":       "Baseline 2024",
    "2034":           "2034",
    "highev":         "High EV",
    "elektrificatie": "Volledige elektrificatie",
}

# Vertaling van interne profielnamen (uit de Fluvius-bestanden) naar leesbare labels
PROFIEL_LABELS = {
    "zonder_middelen":      "Geen ZP/WP/EV",
    "met_EV":               "EV",
    "met_WP":               "WP",
    "met_WP_met_EV":        "WP + EV",
    "met_ZP":               "ZP",
    "met_EV_met_ZP":        "ZP + EV",
    "met_WP_met_ZP":        "ZP + WP",
    "met_WP_met_EV_met_ZP": "ZP + WP + EV",
}

# Vaste volgorde van profielen in outputs en plots (van goedkoopst naar duurste voor de leverancier)
PROFIEL_VOLGORDE = [
    "Geen ZP/WP/EV", "EV", "WP", "WP + EV",
    "ZP", "ZP + EV", "ZP + WP", "ZP + WP + EV",
]

# Labels voor de vier kostenkwartielen (van goedkoopste naar duurste klant)
KWARTIELEN = ["Goedkoopste 25%", "25%-50%", "50%-75%", "Duurste 25%"]


# ============================================================
# STAP 1 - EPEX-PRIJZEN LADEN
# ============================================================

def laad_epex(bestand):
    """
    Laadt een Ember-prijsbestand en maakt het klaar voor koppeling
    aan Fluvius-verbruiksdata. Wordt aangeroepen voor zowel 2024
    (referentiejaar) als 2022 (crisisscenario). De koppeling met de
    Fluvius-data gebeurt op maand/dag/uur in lokale tijd, zodat
    2022-prijzen aan 2024-verbruiken gekoppeld kunnen worden.
    """
    print(f"  EPEX laden: {bestand}")

    # Lees het CSV-bestand in als een pandas DataFrame
    ember = pd.read_csv(bestand)

    # Hernoem de kolommen naar interne namen voor consistentie
    ember = ember.rename(columns={
        "Datetime (Local)": "timestamp_lokaal",  # tijdstempel in lokale Belgische tijd
        "Price (EUR/MWh)":  "Prijs_EUR_MWh",     # groothandelsprijs in EUR per MWh
    })

    # Zet de tijdstempel om naar een datetime-object zodat we datum en uur kunnen extraheren
    ember["timestamp_lokaal"] = pd.to_datetime(ember["timestamp_lokaal"])

    # Zet de prijskolom om naar numerieke waarden; ongeldige waarden worden NaN
    ember["Prijs_EUR_MWh"]    = pd.to_numeric(ember["Prijs_EUR_MWh"], errors="coerce")

    # Bereken de prijs per kWh (= prijs per MWh / 1000) voor gebruik in de tariefberekeningen
    ember["Prijs_EUR_KWh"]    = ember["Prijs_EUR_MWh"] / 1000

    # Extraheer maand, dag en uur als aparte kolommen voor de koppeling met Fluvius-data
    ember["maand"] = ember["timestamp_lokaal"].dt.month
    ember["dag"]   = ember["timestamp_lokaal"].dt.day
    ember["uur"]   = ember["timestamp_lokaal"].dt.hour

    # Wintertijdovergang (oktober): de klok springt terug van 03:00 naar 02:00,
    # waardoor het uur 02:00 twee keer voorkomt in de lokale tijdreeks.
    # Ember bevat daardoor twee rijen voor hetzelfde maand/dag/uur.
    # Oplossing: groepeer op maand/dag/uur en neem het gemiddelde van de twee rijen.
    ember = ember.groupby(["maand", "dag", "uur"]).agg(
        Prijs_EUR_MWh=("Prijs_EUR_MWh", "mean"),
        Prijs_EUR_KWh=("Prijs_EUR_KWh", "mean"),
    ).reset_index()

    # Zomertijdovergang (27 maart 02:00): de klok springt vooruit van 02:00 naar 03:00,
    # waardoor het uur 02:00 niet bestaat in de lokale tijdreeks.
    # Als dit uur ontbreekt in de Ember-data, interpoleer het als gemiddelde van de buren.
    alle_combis = set(zip(ember["maand"], ember["dag"], ember["uur"]))  # alle aanwezige combinaties
    if (3, 27, 2) not in alle_combis:  # controleer of 27 maart 02:00 ontbreekt
        # Zoek de rij voor het uur ervoor (01:00) en het uur erna (03:00)
        rij_voor = ember[(ember["maand"] == 3) & (ember["dag"] == 27) & (ember["uur"] == 1)]
        rij_na   = ember[(ember["maand"] == 3) & (ember["dag"] == 27) & (ember["uur"] == 3)]
        if len(rij_voor) > 0 and len(rij_na) > 0:
            # Bereken het gemiddelde van de twee aangrenzende uurprijzen
            mwh = (rij_voor["Prijs_EUR_MWh"].values[0] + rij_na["Prijs_EUR_MWh"].values[0]) / 2
            # Voeg het geïnterpoleerde uur toe aan het DataFrame
            ember = pd.concat([ember, pd.DataFrame([{
                "maand": 3, "dag": 27, "uur": 2,
                "Prijs_EUR_MWh": mwh, "Prijs_EUR_KWh": mwh / 1000,
            }])], ignore_index=True)
            print(f"    Zomertijduur 27/3 02:00 geïnterpoleerd ({mwh:.2f} EUR/MWh)")

    # Bereken de mediane jaarprijs als fallback voor uren zonder prijskoppeling
    mediaan = ember["Prijs_EUR_MWh"].median()
    print(f"    {len(ember)} uurprijzen | mediaan: {mediaan:.2f} EUR/MWh")
    return ember, mediaan


# ============================================================
# STAP 2 - INDIVIDUELE KOSTENPOSITIE: ALLE 2.400 FLUVIUS-METERS
# ============================================================

def bereken_kostenpositie_2400(ember, mediaan):
    """
    Berekent de individuele leveranciersinkoopkost voor alle 2.400
    Fluvius-meters (300 per profieltype x 8 profielen).
    """
    print("  Individuele kostenpositie berekenen (2.400 meters)...")

    alle_delen = []
    for profiel_label, bestand in FLUVIUS_2400.items():
        # Lees het profielbestand in
        df = pd.read_csv(bestand)

        # Verwijder een eventuele eerste kolom met rij-indices (artefact van eerdere opslag)
        if df.columns[0] not in ["EAN_ID", "Datum"]:
            df = df.iloc[:, 1:]

        # Zet de tijdstempel om van UTC naar Belgische lokale tijd (CET/CEST)
        # tz_localize(None) verwijdert de tijdzone-info na de conversie
        df["Datum_Startuur"] = pd.to_datetime(
            df["Datum_Startuur"], utc=True
        ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)

        # Zet afname- en injectievolumes om naar numeriek; vul ontbrekende waarden met 0
        # clip(lower=0) zorgt dat negatieve waarden (meetfouten) op 0 worden gezet
        df["Volume_Afname_KWh"]   = pd.to_numeric(
            df["Volume_Afname_KWh"], errors="coerce").fillna(0).clip(lower=0)
        df["Volume_Injectie_KWh"] = pd.to_numeric(
            df["Volume_Injectie_KWh"], errors="coerce").fillna(0).clip(lower=0)

        # Klanten zonder PV-installatie kunnen niet injecteren: zet injectie op 0
        df.loc[df["PV_Installatie_Indicator"] == 0, "Volume_Injectie_KWh"] = 0.0

        # Verwijder rijen met een ongeldige tijdstempel (NaT, door zomertijdovergang)
        df = df.dropna(subset=["Datum_Startuur"]).copy()

        # Extraheer maand, dag en uur voor de koppeling met EPEX-prijzen
        df["maand"] = df["Datum_Startuur"].dt.month
        df["dag"]   = df["Datum_Startuur"].dt.day
        df["uur"]   = df["Datum_Startuur"].dt.hour
        df["Profiel_Label"] = profiel_label  # voeg de profielnaam toe als kolom
        alle_delen.append(df)
        print(f"    {profiel_label}: {df['EAN_ID'].nunique()} meters")

    # Voeg alle 8 profielbestanden samen tot één groot DataFrame (2.400 meters)
    data_2400 = pd.concat(alle_delen, ignore_index=True)

    # Koppel de EPEX-uurprijzen aan de Fluvius-data via maand/dag/uur
    # how="left" zorgt dat alle Fluvius-rijen behouden blijven, ook zonder prijskoppeling
    data_2400 = data_2400.merge(
        ember[["maand", "dag", "uur", "Prijs_EUR_KWh", "Prijs_EUR_MWh"]],
        on=["maand", "dag", "uur"], how="left",
    )

    # Vul ontbrekende prijzen op met de mediane jaarprijs als fallback
    data_2400["Prijs_EUR_KWh"] = data_2400["Prijs_EUR_KWh"].fillna(mediaan / 1000)
    data_2400["Prijs_EUR_MWh"] = data_2400["Prijs_EUR_MWh"].fillna(mediaan)

    # Bereken het individuele leverancierstarief per meter
    records = []
    for (ean_id, profiel_label), kd in data_2400.groupby(["EAN_ID", "Profiel_Label"]):
        tot_afn = kd["Volume_Afname_KWh"].sum()   # totaal jaarlijks afnamevolume (kWh)
        tot_inj = kd["Volume_Injectie_KWh"].sum() # totaal jaarlijks injectievolume (kWh)

        if tot_afn == 0:
            continue  # sla meters zonder afname over (niet relevant voor de analyse)

        # Gewogen gemiddelde EPEX-prijs op afname-uren: uren met meer verbruik wegen zwaarder
        # Dit geeft de gemiddelde inkoopkost van de leverancier voor deze klant
        gem_afn        = (kd["Volume_Afname_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_afn
        # Voeg de risicopremie toe: dit is het individuele leverancierstarief excl. btw
        ind_tarief_afn = gem_afn * (1 + RISICO_PREMIE)

        if tot_inj > 0:
            # Gewogen gemiddelde EPEX-prijs op injectie-uren
            gem_inj        = (kd["Volume_Injectie_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_inj
            # Trek de risicopremie af: de leverancier betaalt iets minder dan de marktprijs
            ind_tarief_inj = gem_inj * (1 - RISICO_PREMIE)
        else:
            ind_tarief_inj = np.nan  # geen injectie = geen injectietarief

        records.append({
            "EAN_ID":              ean_id,
            "Profiel_Label":       profiel_label,
            "Afname_KWh":          round(tot_afn, 2),
            "Injectie_KWh":        round(tot_inj, 2),
            "Ind_Tarief_Afname":   round(ind_tarief_afn,  6),  # EUR/kWh, excl. btw
            "Ind_Tarief_Injectie": round(ind_tarief_inj,  6) if not np.isnan(ind_tarief_inj) else np.nan,
        })

    klant_df = pd.DataFrame(records)

    # Bereken het portfoliogemiddelde afnametarief (volume-gewogen over alle 2.400 meters)
    # Dit is het vaste tarief dat de leverancier zou aanrekenen als hij alle 2.400 meters beheert
    gem_portfolio_afn = (
        (klant_df["Afname_KWh"] * klant_df["Ind_Tarief_Afname"]).sum()
        / klant_df["Afname_KWh"].sum()
    )

    # Bereken het portfoliogemiddelde injectietarief (enkel voor meters met injectie)
    inj_sub = klant_df[klant_df["Injectie_KWh"] > 0].dropna(subset=["Ind_Tarief_Injectie"])
    gem_portfolio_inj = (
        (inj_sub["Injectie_KWh"] * inj_sub["Ind_Tarief_Injectie"]).sum()
        / inj_sub["Injectie_KWh"].sum()
    ) if len(inj_sub) > 0 else 0.0

    # Voeg de portfoliogemiddelden toe als kolommen (identiek voor alle meters)
    klant_df["Portfolio_Afname_excl_btw"] = round(gem_portfolio_afn, 6)
    klant_df["Portfolio_Injectie"]        = round(gem_portfolio_inj, 6)

    print(f"    Portfoliogemiddelde afname (excl. btw): {gem_portfolio_afn*100:.4f} ct/kWh")
    print(f"    Portfoliogemiddelde injectie:           {gem_portfolio_inj*100:.4f} ct/kWh")

    # Bereken de kostenkwartielen: verdeel alle 2.400 meters in 4 gelijke groepen
    # op basis van hun individueel afnametarief (van goedkoopst naar duurste)
    klant_sorted = klant_df.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)  # totaal aantal meters (= 2.400)

    # Wijs elk meter toe aan een kwartiel via pd.cut op de rij-index
    klant_sorted["Kwartiel"] = pd.cut(
        klant_sorted.index,
        bins=[0, N//4, N//2, 3*N//4, N],  # grenzen van de vier kwartielen
        labels=KWARTIELEN,
        include_lowest=True,  # zorg dat de laagste waarde ook meegenomen wordt
    )

    # Bereken de profielverdeling (%) per kwartiel via een kruistabel
    # normalize="index" zorgt dat de percentages per rij (= per kwartiel) optellen tot 100%
    kwartiel_df = (
        pd.crosstab(
            klant_sorted["Kwartiel"],
            klant_sorted["Profiel_Label"],
            normalize="index"
        ) * 100
    ).reset_index()

    return klant_df, kwartiel_df


# ============================================================
# STAP 3 - FLUVIUS-DATA LADEN EN KOPPELEN AAN EPEX
# ============================================================

def laad_fluvius(bestand, ember, mediaan, filter_29feb=False):
    """
    Laadt een Fluvius-steekproefbestand (1.000 huishoudens) en koppelt
    de EPEX-uurprijzen via maand/dag/uur in lokale Belgische tijd.
    """
    print(f"  Fluvius laden: {bestand}")

    # Lees het steekproefbestand in als een pandas DataFrame
    fluvius = pd.read_csv(bestand)

    # Tijdstempel van UTC naar Belgische lokale tijd (CET/CEST).
    # De Fluvius-data staat in UTC (herkenbaar aan de Z-suffix in de tijdstempel).
    # Na conversie naar lokale tijd (Europe/Brussels) ontstaat op 27 maart om 02:00
    # een ongeldige tijdstempel (NaT) omdat dat uur niet bestaat door de zomertijdovergang.
    # tz_localize(None) verwijdert de tijdzone-info na de conversie.
    fluvius["Datum_Startuur"] = pd.to_datetime(
        fluvius["Datum_Startuur"], utc=True
    ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)

    # Zet afname- en injectievolumes om naar numeriek; vul ontbrekende waarden met 0
    # clip(lower=0) zorgt dat negatieve waarden (meetfouten) op 0 worden gezet
    fluvius["Volume_Afname_KWh"]   = pd.to_numeric(
        fluvius["Volume_Afname_KWh"], errors="coerce").fillna(0).clip(lower=0)
    fluvius["Volume_Injectie_KWh"] = pd.to_numeric(
        fluvius["Volume_Injectie_KWh"], errors="coerce").fillna(0).clip(lower=0)

    # Klanten zonder PV-installatie kunnen niet injecteren: zet injectie op 0
    fluvius.loc[fluvius["PV_Installatie_Indicator"] == 0, "Volume_Injectie_KWh"] = 0.0

    # Verwijder rijen met een ongeldige tijdstempel (NaT, door de zomertijdovergang op 27 maart)
    # Het jaar 2024 telt daardoor 8.783 uren per klant in plaats van 8.784
    n_voor = len(fluvius)
    fluvius = fluvius.dropna(subset=["Datum_Startuur"]).copy()
    if len(fluvius) < n_voor:
        print(f"    Zomertijd (NaT): {n_voor - len(fluvius)} rijen verwijderd")

    # Crisisscenario: filter rijen van 29 februari 2024 weg
    # want 2022 is geen schrikkeljaar en heeft geen 29 februari
    if filter_29feb:
        n_voor = len(fluvius)
        fluvius = fluvius[
            ~((fluvius["Datum_Startuur"].dt.month == 2) &
              (fluvius["Datum_Startuur"].dt.day == 29))
        ].copy()
        print(f"    29/02 gefilterd: {n_voor - len(fluvius)} rijen verwijderd")

    # Extraheer maand, dag en uur als aparte kolommen voor de koppeling met EPEX-prijzen
    fluvius["maand"] = fluvius["Datum_Startuur"].dt.month
    fluvius["dag"]   = fluvius["Datum_Startuur"].dt.day
    fluvius["uur"]   = fluvius["Datum_Startuur"].dt.hour

    # Koppel de EPEX-uurprijzen aan de Fluvius-data via maand/dag/uur
    # how="left" zorgt dat alle Fluvius-rijen behouden blijven, ook zonder prijskoppeling
    data = fluvius.merge(
        ember[["maand", "dag", "uur", "Prijs_EUR_KWh", "Prijs_EUR_MWh"]],
        on=["maand", "dag", "uur"], how="left",
    )

    # Controleer hoeveel rijen geen prijskoppeling hebben en vul op met de mediane jaarprijs
    n_missing = data["Prijs_EUR_MWh"].isna().sum()
    if n_missing > 0:
        print(f"    {n_missing} ontbrekende prijzen -> mediaan gebruikt")
    data["Prijs_EUR_MWh"] = data["Prijs_EUR_MWh"].fillna(mediaan)
    data["Prijs_EUR_KWh"] = data["Prijs_EUR_KWh"].fillna(mediaan / 1000)

    # Vertaal de interne profielnamen naar leesbare labels
    data["Profiel_Label"] = data["Profiel"].map(PROFIEL_LABELS).fillna(data["Profiel"])

    print(f"    {data['EAN_ID'].nunique()} klanten | {len(data):,} rijen")
    return data


# ============================================================
# STAP 4 - TARIEVEN BEREKENEN PER KLANT
# ============================================================

def bereken_tarieven(data):
    """
    Berekent het vaste portfoliotarief en de individuele tarieven per klant
    op basis van de 1.000-steekproef.

    """
    # Bereken het totale afname- en injectievolume over alle klanten en uren
    tot_afname   = data["Volume_Afname_KWh"].sum()
    tot_injectie = data["Volume_Injectie_KWh"].sum()

    # Portfoliogemiddelde afnametarief: gewogen gemiddelde EPEX over alle afname-uren en klanten
    gem_afn  = (data["Volume_Afname_KWh"] * data["Prijs_EUR_KWh"]).sum() / tot_afname
    # Voeg de risicopremie toe: dit is het vaste portfoliotarief excl. btw
    vast_afn = gem_afn * (1 + RISICO_PREMIE)

    if tot_injectie > 0:
        # Portfoliogemiddelde injectietarief: gewogen gemiddelde EPEX over alle injectie-uren
        gem_inj  = (data["Volume_Injectie_KWh"] * data["Prijs_EUR_KWh"]).sum() / tot_injectie
        # Trek de risicopremie af: de leverancier betaalt iets minder dan de marktprijs voor injectie
        vast_inj = gem_inj * (1 - RISICO_PREMIE)
    else:
        vast_inj = 0.0  # geen injectie in de portfolio

    # Bereken voor elke klant individueel de tarieven en kosten
    records = []
    for (ean_id, profiel), kd in data.groupby(["EAN_ID", "Profiel"]):
        tot_afn_k = kd["Volume_Afname_KWh"].sum()   # jaarlijks afnamevolume van deze klant (kWh)
        tot_inj_k = kd["Volume_Injectie_KWh"].sum() # jaarlijks injectievolume van deze klant (kWh)

        if tot_afn_k == 0:
            continue  # sla klanten zonder afname over

        # Gewogen gemiddelde EPEX-prijs op de afname-uren van deze specifieke klant
        gem_afn_k = (kd["Volume_Afname_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_afn_k

        # Individueel leverancierstarief excl. btw: dit is de rangschikkingsbasis voor de death spiral
        # Klanten met een lager individueel tarief zijn goedkoper voor de leverancier
        ind_tarief_excl = gem_afn_k * (1 + RISICO_PREMIE)

        # Jaarlijkse kost voor de klant op het vaste contract (energiecomponent, excl. btw)
        # = jaarlijks afnamevolume × portfoliogemiddelde tarief
        kost_vast_afn = tot_afn_k * vast_afn

        # Jaarlijkse kost voor de klant op het dynamische contract (energiecomponent, excl. btw)
        # = per uur: afnamevolume × uurlijks dynamisch tarief (op basis van de Eneco tariefformule)
        tarief_dyn_uur = (DYN_ALPHA * kd["Prijs_EUR_MWh"] + DYN_BETA) / 100
        kost_dyn_afn   = (kd["Volume_Afname_KWh"] * tarief_dyn_uur).sum()

        if tot_inj_k > 0:
            # Gewogen gemiddelde EPEX-prijs op de injectie-uren van deze klant
            gem_inj_k      = (kd["Volume_Injectie_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_inj_k
            # Individueel injectietarief excl. btw
            ind_tarief_inj = gem_inj_k * (1 - RISICO_PREMIE)
            # Jaarlijkse injectieopbrengst op het vaste contract
            kost_vast_inj  = tot_inj_k * vast_inj
            # Jaarlijkse injectievergoeding op het dynamische contract
            # Kan negatief zijn bij lage of negatieve EPEX-prijzen:
            # de klant betaalt dan bij aan de onbalanskosten van de leverancier
            vergoeding_dyn = (DYN_ALPHA_INJ * kd["Prijs_EUR_MWh"] - DYN_BETA_INJ) / 100
            kost_dyn_inj   = (kd["Volume_Injectie_KWh"] * vergoeding_dyn).sum()
        else:
            # Geen injectie: alle injectiegerelateerde waarden op nul of NaN zetten
            ind_tarief_inj = np.nan
            kost_vast_inj  = 0.0
            kost_dyn_inj   = 0.0

        records.append({
            "EAN_ID":               ean_id,
            "Profiel":              profiel,
            "Profiel_Label":        PROFIEL_LABELS.get(profiel, profiel),
            "Afname_KWh":           round(tot_afn_k,       2),
            "Injectie_KWh":         round(tot_inj_k,       2),
            # Leveranciersinkoopkost excl. btw: rangschikkingsbasis voor de death spiral
            "Ind_Tarief_Afname":    round(ind_tarief_excl, 6),
            # Portfoliotarief: identiek voor alle klanten in dit scenario
            "Portfolio_Afname_ct":  round(vast_afn * 100,  4),
            "Portfolio_Injectie_ct":round(vast_inj * 100,  4),
            # Jaarlijkse kosten energiecomponent: basis voor de dyn/vast vergelijking
            "Kost_Vast_Afname":     round(kost_vast_afn,   2),
            "Kost_Dyn_Afname":      round(kost_dyn_afn,    2),
            "Opbr_Vast_Injectie":   round(kost_vast_inj,   2),
            "Opbr_Dyn_Injectie":    round(kost_dyn_inj,    2),
            # Boolean: is de afnamecomponent van het dynamisch contract goedkoper dan vast?
            "Dyn_Beter_Afname":     kost_dyn_afn < kost_vast_afn,
        })

    return vast_afn * 100, vast_inj * 100, pd.DataFrame(records)


# ============================================================
# STAP 5 - DEATH SPIRAL SIMULEREN
# ============================================================

def bereken_death_spiral(klant):
    # Sorteer klanten van goedkoopste naar duurste voor de leverancier
    # De goedkoopste klanten vertrekken als eersten in de death spiral
    klant_sorted = klant.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)  # totaal aantal klanten in de portfolio

    records = []
    for stap in range(SPIRAL_STAPPEN + 1):
        # Bereken welk percentage van de klanten al vertrokken is in deze stap
        pct = stap / SPIRAL_STAPPEN  # 0.0, 0.02, 0.04, ..., 1.0

        # Selecteer de achterblijvers: de duurste (1 - pct) × N klanten
        achterblijvers = klant_sorted.iloc[int(N * pct):]

        # Stop de simulatie als er minder dan 10 achterblijvers zijn
        # (te klein voor een zinvol portfoliogemiddelde)
        if len(achterblijvers) < 10:
            break

        # Bereken het totale afnamevolume van de achterblijvers
        tot_afname = achterblijvers["Afname_KWh"].sum()
        if tot_afname == 0:
            break  # stop als er geen afname meer is

        # Bereken het volume-gewogen gemiddelde leverancierstarief van de achterblijvers
        # Dit is het nieuwe vaste portfoliotarief na het vertrek van de goedkoopste klanten
        gem_excl = (
            achterblijvers["Ind_Tarief_Afname"] * achterblijvers["Afname_KWh"]
        ).sum() / tot_afname

        # Converteer naar ct/kWh voor de output
        energie_ct = gem_excl * 100

        records.append({
            "pct_vertrokken": round(pct * 100,  2),  # percentage vertrokken klanten
            "energie_ct":     round(energie_ct,  4),  # vast energietarief achterblijvers (ct/kWh)
        })

    return pd.DataFrame(records)


def bereken_profieldrempels(klant):
    """
    Berekent voor elk klantprofiel op welk punt in de gesorteerde
    portfolio 80% van dat profiel al vertrokken is.
    Gebruikt als stippellijn in de death spiral plot.
    """
    # Sorteer klanten van goedkoopste naar duurste voor de leverancier
    klant_sorted = klant.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)  # totaal aantal klanten

    records = []
    for profiel, sub in klant_sorted.groupby("Profiel_Label"):
        if len(sub) == 0:
            continue

        # Bereken de positie (index) in de gesorteerde portfolio waarop
        # 80% van dit profieltype al vertrokken is (= het 80e percentiel van de indices)
        idx = np.percentile(sub.index.to_numpy(), DREMPEL_PROFIEL * 100)

        records.append({
            "Profiel_Label": profiel,
            # Converteer de index naar een percentage van de totale portfolio
            "pct_drempel":   round((idx + 1) / N * 100, 2),
        })

    # Sorteer op drempelwaarde zodat de stippellijnen in volgorde verschijnen in de plot
    return pd.DataFrame(records).sort_values("pct_drempel")


# ============================================================
# STAP 6+7 - DYNAMISCH VS VAST EN GEVOELIGHEIDSANALYSE
# ============================================================

def bereken_gevoeligheid(data):
    """
    Berekent voor elk volatiliteitsscenario het % klanten waarvoor
    een dynamisch contract goedkoper is dan een vast contract,
    uitsluitend op basis van de energiecomponent.

    De netcomponent en heffingen zijn voor beide contracttypes identiek
    en vallen weg in de vergelijking, dus worden hier niet meegenomen.

    De EPEX-prijzen worden geschaald rond hun jaargemiddelde:
      geschaalde prijs = gemiddelde + (originele prijs - gemiddelde) × factor

    Bij factor > 1 zijn de uitschieters groter maar het gemiddelde
    en dus het vaste energietarief blijft nagenoeg gelijk. Enkel de
    spreiding vergroot, waardoor het dynamisch contract aantrekkelijker
    wordt voor klanten met een gunstig verbruiksprofiel.
    """
    # Bereken het jaargemiddelde van de EPEX-prijzen (anker voor de schaling)
    gem_prijs = data["Prijs_EUR_MWh"].mean()
    records   = []

    for vol_naam, vol_factor in VOLATILITEIT_SCENARIOS.items():
        # Maak een kopie van de data voor dit volatiliteitsscenario
        data_vol = data.copy()

        # Schaal de prijzen rond het jaargemiddelde:
        # geschaalde prijs = gemiddelde + (originele prijs - gemiddelde) × factor
        # Bij factor = 1: ongewijzigd; bij factor = 2: dubbele spreiding rond hetzelfde gemiddelde
        data_vol["Prijs_EUR_MWh"] = gem_prijs + (data["Prijs_EUR_MWh"] - gem_prijs) * vol_factor
        data_vol["Prijs_EUR_KWh"] = data_vol["Prijs_EUR_MWh"] / 1000

        # Bereken het nieuwe vaste portfoliotarief voor dit volatiliteitsscenario
        tot_afname = data_vol["Volume_Afname_KWh"].sum()
        gem_afn    = (data_vol["Volume_Afname_KWh"] * data_vol["Prijs_EUR_KWh"]).sum() / tot_afname
        vast_afn   = gem_afn * (1 + RISICO_PREMIE)  # excl. btw

        # Bereken per klant of het dynamische contract goedkoper is dan het vaste contract
        klant_records = []
        for (ean_id, profiel), kd in data_vol.groupby(["EAN_ID", "Profiel"]):
            tot_afn_k = kd["Volume_Afname_KWh"].sum()
            if tot_afn_k == 0:
                continue

            # Jaarlijkse kost op het vaste contract (energiecomponent)
            kost_vast      = tot_afn_k * vast_afn

            # Jaarlijkse kost op het dynamische contract (energiecomponent)
            tarief_dyn_uur = (DYN_ALPHA * kd["Prijs_EUR_MWh"] + DYN_BETA) / 100
            kost_dyn       = (kd["Volume_Afname_KWh"] * tarief_dyn_uur).sum()

            klant_records.append({
                "Profiel_Label":    PROFIEL_LABELS.get(profiel, profiel),
                "dyn_beter_afname": kost_dyn < kost_vast,  # True als dynamisch goedkoper is
            })

        klant_df = pd.DataFrame(klant_records)

        # Bereken per profieltype het percentage klanten waarvoor dynamisch goedkoper is
        for profiel in PROFIEL_VOLGORDE:
            sub = klant_df[klant_df["Profiel_Label"] == profiel]
            if len(sub) == 0:
                continue
            records.append({
                "volatiliteit":  vol_naam,
                "vol_factor":    vol_factor,
                "profiel":       profiel,
                "vast_ct_incl":  round(vast_afn * 100, 2),          # vast tarief in ct/kWh
                "pct_dyn_beter": round(sub["dyn_beter_afname"].mean() * 100, 1),  # % klanten dyn beter
            })

    return pd.DataFrame(records)


# ============================================================
# HOOFDPROGRAMMA
# ============================================================

print("=" * 60)
print("SIMULATIE STARTEN")
print("=" * 60)

# Stap 1: Laad de EPEX-prijzen voor 2024 (referentiejaar) en 2022 (crisisscenario)
print("\n[1/4] EPEX-prijzen laden...")
ember_2024, mediaan_2024 = laad_epex("Belgium_2024_MWh.csv")
ember_2022, mediaan_2022 = laad_epex("Belgium_2022_MWh.csv")

# Stap 2: Bereken de individuele kostenpositie voor alle 2.400 Fluvius-meters
# Dit is de basis voor de puntenwolk en kwartielverdeling in de thesis
print("\n[2/4] Individuele kostenpositie (2.400 meters)...")
klant_2400, kwartiel_2400 = bereken_kostenpositie_2400(ember_2024, mediaan_2024)
klant_2400.to_csv("individuele_tarieven_2400.csv", index=False)    # sla op als CSV
kwartiel_2400.to_csv("kostenkwartielen_2400.csv", index=False)      # sla op als CSV
print(f"  Opgeslagen: individuele_tarieven_2400.csv ({len(klant_2400)} meters)")
print(f"  Opgeslagen: kostenkwartielen_2400.csv")

# Stap 3+4+5: Bereken per scenario de tarieven, death spiral en dyn/vast vergelijking
print("\n[3/4] Tarieven, death spiral en dyn/vast per scenario...")

klant_data   = {}   # bewaar de klantresultaten per scenario (voor later gebruik)
spiral_data  = {}   # bewaar de death spiral curves per scenario
dyn_vast_res = []   # bewaar de dyn/vast resultaten voor alle scenario's samen

for scenario, bestand in FLUVIUS_BESTANDEN.items():
    label = SCENARIO_LABELS[scenario]
    print(f"\n  Scenario: {label}")

    # Laad de Fluvius-steekproef en koppel de 2024-EPEX-prijzen
    data = laad_fluvius(bestand, ember_2024, mediaan_2024)

    # Bereken de tarieven per klant en het portfoliogemiddelde
    vast_ct, vast_inj_ct, klant_df = bereken_tarieven(data)
    print(f"    Energietarief vast (excl. btw):  {vast_ct:.4f} ct/kWh")
    print(f"    Injectietarief vast (excl. btw): {vast_inj_ct:.4f} ct/kWh")

    # Voeg het scenario-label toe en sla de klantresultaten op
    klant_df["Scenario"] = label
    klant_df.to_csv(f"tarieven_{scenario}.csv", index=False)
    klant_data[scenario] = klant_df

    # Simuleer de death spiral: laat de goedkoopste klanten stap voor stap vertrekken
    spiral = bereken_death_spiral(klant_df)
    spiral.to_csv(f"spiral_{scenario}.csv", index=False)
    spiral_data[scenario] = spiral

    # Print het begin- en eindtarief van de death spiral curve
    s_e = spiral["energie_ct"].iloc[0]   # starttarief (0% vertrokken)
    e_e = spiral["energie_ct"].iloc[-1]  # eindtarief (98% vertrokken)
    print(f"    Death spiral: {s_e:.2f} -> {e_e:.2f} ct/kWh ({(e_e-s_e)/s_e*100:+.1f}%)")

    # Bereken per profieltype het percentage klanten waarvoor dynamisch goedkoper is
    for profiel in PROFIEL_VOLGORDE:
        sub = klant_df[klant_df["Profiel_Label"] == profiel]
        if len(sub) == 0:
            continue
        pct = sub["Dyn_Beter_Afname"].mean() * 100  # % klanten waarvoor dyn beter is
        dyn_vast_res.append({
            "scenario":      label,
            "profiel":       profiel,
            "vast_ct_incl":  round(vast_ct, 2),
            "pct_dyn_beter": round(pct, 1),
        })

# Bereken de profieldrempels voor de baseline plot (stippellijnen in de death spiral grafiek)
drempels = bereken_profieldrempels(klant_data["baseline"])
drempels.to_csv("spiral_profieldrempels.csv", index=False)

# Crisisscenario 2022: gebruik dezelfde klantenmix als de baseline maar met 2022-prijzen
# filter_29feb=True: verwijder 29 februari (bestaat niet in 2022)
print("\n  Crisisscenario: Baseline met 2022-prijzen")
data_crisis   = laad_fluvius(FLUVIUS_BESTANDEN["baseline"], ember_2022, mediaan_2022, filter_29feb=True)
_, _, klant_c = bereken_tarieven(data_crisis)
spiral_c      = bereken_death_spiral(klant_c)
spiral_c.to_csv("spiral_baseline_2022.csv", index=False)
s_c = spiral_c["energie_ct"].iloc[0]
e_c = spiral_c["energie_ct"].iloc[-1]
print(f"    Death spiral: {s_c:.2f} -> {e_c:.2f} ct/kWh ({(e_c-s_c)/s_c*100:+.1f}%)")

# Sla de volledige dyn/vast resultaten op als één CSV
pd.DataFrame(dyn_vast_res).to_csv("dyn_vs_vast.csv", index=False)

# Stap 7: Gevoeligheidsanalyse op de 2034-steekproef (alle profieltypes voldoende vertegenwoordigd)
print("\n[4/4] Gevoeligheidsanalyse volatiliteit (2034-steekproef)...")
data_2034    = laad_fluvius(FLUVIUS_BESTANDEN["2034"], ember_2024, mediaan_2024)
gevoeligheid = bereken_gevoeligheid(data_2034)
gevoeligheid.to_csv("gevoeligheid_dyn_vast.csv", index=False)

# Print een overzicht van alle resultaten
print("\n" + "=" * 68)
print("OVERZICHT RESULTATEN")
print("=" * 68)

print("\nDeath spiral per scenario (leverancierstarief excl. btw):")
print(f"  {'Scenario':<30} {'Start ct/kWh':>13} {'Eind ct/kWh':>12} {'Stijging':>10}")
print("  " + "-" * 68)
for scenario, spiral in spiral_data.items():
    s = spiral["energie_ct"].iloc[0]
    e = spiral["energie_ct"].iloc[-1]
    print(f"  {SCENARIO_LABELS[scenario]:<30} {s:>12.2f}  {e:>11.2f}  {(e-s)/s*100:>+9.1f}%")
s_c = spiral_c["energie_ct"].iloc[0]
e_c = spiral_c["energie_ct"].iloc[-1]
print(f"  {'Crisisscenario 2022':<30} {s_c:>12.2f}  {e_c:>11.2f}  {(e_c-s_c)/s_c*100:>+9.1f}%")

print("\nDynamisch vs vast - % klanten dyn beter per scenario (energiecomponent):")
scenario_namen = list(SCENARIO_LABELS.values())
print(f"  {'Profiel':<20} " + "".join(f"{s:>22}" for s in scenario_namen))
print("  " + "-" * (20 + 22 * len(scenario_namen)))
for profiel in PROFIEL_VOLGORDE:
    rij = f"  {profiel:<20}"
    for naam in scenario_namen:
        match = [r for r in dyn_vast_res if r["scenario"] == naam and r["profiel"] == profiel]
        rij += f"{match[0]['pct_dyn_beter']:>21.1f}%" if match else f"{'—':>22}"
    print(rij)

print("\nAlle outputbestanden aangemaakt.")
print("Voer de plotbestanden uit om de grafieken te genereren.")